# PoliMillionaire Model Tests

NLP assignment, 2025/26.

| Student | Polimi email |
| --- | --- |
| Juan Martin Sanchez | juanmartin.sancehz@mail.polimi.it |
| Camilo Andres Martinez Mejia | camiloandres1.martinez@mail.polimi.it |
| Reinaldo Toledo | reinaldo.toledo@mail.polimi.it |

Video: add link here.

Academic integrity: add required statement here.


## Run Order

Pick the local models in `MODELS_TO_TEST`, then run downward.


In [1]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))

print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])


Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Settings

Add or remove models here. They run one at a time.


In [2]:
INSTALL_DEPS = False

MODELS_TO_TEST = [
    {"label": "Gemma 4 E2B", "model_id": "google/gemma-4-E2B-it", "run": True},
    {"label": "Gemma 4 E4B", "model_id": "google/gemma-4-E4B-it", "run": True},
]

print("Install deps:", INSTALL_DEPS)
print("Models:")
for item in MODELS_TO_TEST:
    print("-", item["label"], item["model_id"], "run=", item["run"])


Install deps: False
Models:
- Gemma 4 E2B google/gemma-4-E2B-it run= True
- Gemma 4 E4B google/gemma-4-E4B-it run= True


In [3]:
if INSTALL_DEPS:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])
else:
    print("Install skipped.")


Install skipped.


## Environment

Checks the notebook kernel before loading any model.


In [4]:
import importlib.util
from packaging.version import Version

print("Python:", sys.executable)

for package_name in ["torch", "transformers", "accelerate"]:
    available = importlib.util.find_spec(package_name) is not None
    version = None
    if available:
        try:
            module = __import__(package_name)
            version = getattr(module, "__version__", "installed")
        except Exception as exc:
            version = f"import error: {exc}"
    print(f"{package_name}: {available} {version or ''}")

if importlib.util.find_spec("transformers") is not None:
    import transformers

    if Version(transformers.__version__) < Version("5.7.0"):
        print("Transformers is too old for Gemma 4. Set INSTALL_DEPS=True, run install, restart.")


Python: c:\Users\sjuan\Anaconda3\envs\gen\python.exe
torch: True 2.9.0+cu130
transformers: True 5.7.0
accelerate: True 1.4.0


## Test Question

Same question for every selected model.


In [5]:
from polimillionaire.types import AnswerOption, Question

sample_question = Question(
    id=1,
    text="What is 2 + 2?",
    options=[
        AnswerOption(0, "3"),
        AnswerOption(1, "4"),
        AnswerOption(2, "5"),
        AnswerOption(3, "22"),
    ],
    level=1,
)

print(sample_question.text)
for option in sample_question.options:
    print(option.id, option.text)


What is 2 + 2?
0 3
1 4
2 5
3 22


## Run Selected Models

Each model is unloaded before the next one starts.


In [6]:
import gc

from polimillionaire.strategies import GemmaLLMConfig, GemmaStrategy


def clear_model_memory():
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "mps") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except Exception as exc:
        print("Memory cleanup warning:", exc)


def run_model_test(model_id: str, label: str):
    config = GemmaLLMConfig(
        model_id=model_id,
        inference_backend="auto_model",
        device_map="auto",
        dtype="auto",
        max_new_tokens=8,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        seed=42,
        generation_max_time_seconds=18.0,
        timeout_seconds=120.0,
    )
    model = None
    try:
        model = GemmaStrategy(model_config=config)
        prediction = model.answer(sample_question)

        print()
        print(label)
        print("model:", model_id)
        print("option:", prediction.option_id, prediction.answer_text)
        print("confidence:", prediction.confidence)
        print("reason:", prediction.reasoning)
        print("fallback:", prediction.metadata.get("fallback"))
        print("device:", prediction.metadata.get("device"))
        print("raw:", prediction.metadata.get("raw_text"))

        if prediction.metadata.get("fallback"):
            raise RuntimeError(f"{label} output was not parsed cleanly.")
        return {
            "label": label,
            "model_id": model_id,
            "option_id": prediction.option_id,
            "answer_text": prediction.answer_text,
            "confidence": prediction.confidence,
            "reason": prediction.reasoning,
            "fallback": prediction.metadata.get("fallback"),
        }
    finally:
        del model
        clear_model_memory()
        print("Cleared model memory after", label)


model_results = []
for item in MODELS_TO_TEST:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue
    model_results.append(run_model_test(item["model_id"], item["label"]))

model_results


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]


Gemma 4 E2B
model: google/gemma-4-E2B-it
option: 1 4
confidence: 1.0
reason: Basic arithmetic fact
fallback: False
raw: {"option_id": 1, "confidence": 1.0, "reason": "Basic arithmetic fact"}
Cleared model memory after Gemma 4 E2B


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

c:\Users\sjuan\Anaconda3\envs\gen\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home\hub\models--google--gemma-4-E4B-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some parameters are on the meta device because they were offloaded to the cpu.



Gemma 4 E4B
model: google/gemma-4-E4B-it
option: 1 4
confidence: 1.0
reason: Two plus two equals four.
fallback: False
raw: {"option_id": 1, "confidence": 1.0, "reason": "Two plus two equals four."}
Cleared model memory after Gemma 4 E4B


[{'label': 'Gemma 4 E2B',
  'model_id': 'google/gemma-4-E2B-it',
  'option_id': 1,
  'answer_text': '4',
  'confidence': 1.0,
  'reason': 'Basic arithmetic fact',
  'fallback': False},
 {'label': 'Gemma 4 E4B',
  'model_id': 'google/gemma-4-E4B-it',
  'option_id': 1,
  'answer_text': '4',
  'confidence': 1.0,
  'reason': 'Two plus two equals four.',
  'fallback': False}]

## Saved Logs

Optional check of previous Gemma live runs.


In [7]:
from polimillionaire.runner import load_jsonl, summarize_attempts

log_dir = REPO_ROOT / "results" / "runs"
gemma_logs = sorted(log_dir.glob("gemma*.jsonl"), key=lambda path: path.stat().st_mtime)

for path in gemma_logs:
    print(path.name)

if gemma_logs:
    rows = load_jsonl(gemma_logs[-1])
    print("latest:", gemma_logs[-1].name)
    print(summarize_attempts(rows))
else:
    print("No Gemma logs found.")


No Gemma logs found.


## Done

Run cleanly, keep useful outputs, export to HTML.
